# Junior results

How the Junior networks learn: per-epoch training curves and a per-state analysis of the selected
model, for the best trial of each Junior HPO study (hard labels, soft labels with τ ∈ {0.005, 0.01, 0.02}),
and the hyperparameter search itself (section C).

The best trials were retrained with their HPO settings (`slurm/unicluster/junior_retrain.sh`), so that
`history.csv` also logs top-20 accuracy and the run writes per-state predictions. GPU training is not
bit-for-bit reproducible, so the numbers differ slightly from the HPO trials.

**Splits.** Everything here is on the **validation** split: section A per epoch, section B for the
selected model (lowest validation KL). The test split is left for the final report, so design choices
made from these plots do not leak into it. Validation is not untouched either: it drove early stopping
and the HPO objective, so its numbers are optimistic, most of all for the KL that was optimised.

**Data.** Sections A and B read the retrained runs, section C the Optuna studies. A local copy needs
two rsyncs (weights not needed):

```
results/junior/junior_training/<study>/   history.csv, val_predictions.npz, config.yaml
                                          (+ policy_weights.pt, test_metrics.json, job logs)
results/junior/<study>/                   study.log (Optuna journal), trials.csv

rsync -av --exclude '*.pt' --exclude hydra <cluster>:<repo>/results/junior/junior_training/ \
    results/junior/junior_training/
rsync -av --include 'junior_*/' --include study.log --include trials.csv --exclude '*' \
    <cluster>:<repo>/results/junior/ results/junior/
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import yaml
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from imitation.hpo import TRIALS_FILE, open_study

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Paths ──
RESULTS = ROOT / "results/junior"
RUNS_DIR = RESULTS / "junior_training"
RUNS = {  # label -> run directory (retrained best trial of the Junior HPO study)
    "hard": RUNS_DIR / "junior_hard",
    "soft τ=0.005": RUNS_DIR / "junior_soft_t0.005",
    "soft τ=0.01": RUNS_DIR / "junior_soft_t0.01",
    "soft τ=0.02": RUNS_DIR / "junior_soft_t0.02",
}
STUDIES = {label: RESULTS / run.name for label, run in RUNS.items()}  # Optuna study per dataset
HPO_CONFIG = ROOT / "configs/rllib/pretrain_hpo.yaml"                 # search space of the studies
FIGURES = ROOT / "experiments/figures/junior_results"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── Constants ──
MONITOR = "val_kl"         # model selection metric (imitation.monitor)
K_MARKS = (1, 5, 20, 50)   # k of the top-k simulation rollouts (results/junior/<study>/best/rollout_top<k>/)

# ── Palette: one fixed categorical slot per dataset ──
DATASET_COLORS = dict(zip(RUNS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"


def style(ax: plt.Axes) -> None:
    ax.set_facecolor(SURFACE)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, length=0)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def save(fig: plt.Figure, name: str) -> None:
    """Write the figure to experiments/figures/junior_results/<name>.png."""
    fig.savefig(FIGURES / f"{name}.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")


def dataset_handles(*extra: Line2D) -> list[Line2D]:
    """Legend entries: one line per dataset, then `extra` (line styles and markers in neutral ink)."""
    return [Line2D([0], [0], color=c, linewidth=2, label=label) for label, c in DATASET_COLORS.items()] + list(extra)


BEST_HANDLE = Line2D([0], [0], marker="o", color="none", markerfacecolor=INK_MUTED, label="selected epoch")


def finish(fig: plt.Figure, title: str, handles: list[Line2D], name: str) -> None:
    fig.legend(handles=handles, frameon=False, ncols=len(handles), loc="upper right", labelcolor=INK, fontsize=9)
    fig.suptitle(title, x=0.01, ha="left", color=INK, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.55 / fig.get_figheight()))
    save(fig, name)
    plt.show()

In [ ]:
histories = {label: pd.read_csv(d / "history.csv") for label, d in RUNS.items()}
configs = {label: yaml.safe_load((d / "config.yaml").read_text()) for label, d in RUNS.items()}
best = {label: h.loc[h[MONITOR].idxmin()] for label, h in histories.items()}

predictions = {}
for label, d in RUNS.items():
    with np.load(d / "val_predictions.npz") as f:
        predictions[label] = {key: f[key] for key in f.files}
N_ACTIONS = int(predictions["hard"]["n_actions"])
TOLERANCE = float(predictions["hard"]["good_action_tolerance"])


def state_frame(p: dict[str, np.ndarray]) -> pd.DataFrame:
    """One row per val state; regret = rho(argmax) − rho(Tutor's action), inf if the argmax is invalid."""
    df = pd.DataFrame({key: value for key, value in p.items() if value.ndim == 1})
    df["match"] = df["pred"] == df["tutor_action"]
    df["pred_valid"] = np.isfinite(df["rho_pred"])
    df["tutor_valid"] = np.isfinite(df["rho_tutor"])
    df["regret"] = df["rho_pred"] - df["rho_tutor"]
    return df


states = {label: state_frame(p) for label, p in predictions.items()}

pd.DataFrame({
    label: {
        "epochs run": len(h),
        "epoch cap": configs[label]["imitation"]["epochs"],
        "patience": configs[label]["imitation"]["patience"],
        "selected epoch": int(best[label]["epoch"]),
        "top-20 per epoch logged": "val_top20" in h,
        "val states": len(states[label]),
        "states without valid Tutor action": int((~states[label]["tutor_valid"]).sum()),
    }
    for label, h in histories.items()
}).T

# A. Training progress (validation, per epoch)

## Cross-entropy and KL

Left: validation cross-entropy, with the mean entropy of each dataset's validation targets as dotted
line of the same color: the lower bound of the CE (0 for hard labels). Right: KL = CE − entropy, the
part of the CE the network can still reduce and the model selection metric. Unlike the CE, the KL is
comparable between datasets. Markers: selected epoch (lowest validation KL).

In [ ]:
fig, (ax_ce, ax_kl) = plt.subplots(1, 2, figsize=(12, 4.2), facecolor=SURFACE)
for label, h in histories.items():
    color, b = DATASET_COLORS[label], best[label]
    ax_ce.plot(h["epoch"], h["val_loss"], color=color, linewidth=2)
    ax_ce.axhline(h["val_entropy"].iloc[0], color=color, linewidth=1.5, linestyle=":")
    ax_ce.scatter([b["epoch"]], [b["val_loss"]], s=50, color=color, edgecolors=SURFACE, linewidths=2, zorder=3)
    ax_kl.plot(h["epoch"], h["val_kl"], color=color, linewidth=2)
    ax_kl.scatter([b["epoch"]], [b["val_kl"]], s=50, color=color, edgecolors=SURFACE, linewidths=2, zorder=3)
for ax, ylabel in ((ax_ce, "Validation cross-entropy [nats]"), (ax_kl, "Validation KL = CE − H(target) [nats]")):
    ax.set_xlabel("Epoch", color=INK_MUTED)
    ax.set_ylabel(ylabel, color=INK_MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    style(ax)
handles = dataset_handles(Line2D([0], [0], color=INK_MUTED, linewidth=1.5, linestyle=":", label="target entropy"),
                          BEST_HANDLE)
finish(fig, "Validation cross-entropy and KL", handles, "ce_kl")

## Train vs. validation cross-entropy

One panel per dataset. A validation CE that turns up while the train CE keeps falling is overfitting.
The train CE is the running mean over each epoch while the weights still change (and with dropout
active), so it lags behind and sits slightly high, most visibly in the first epochs. The dashed line is
the validation target entropy; the train targets have about the same.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), facecolor=SURFACE, sharex=True)
for ax, (label, h) in zip(axes.flat, histories.items()):
    color, b = DATASET_COLORS[label], best[label]
    ax.plot(h["epoch"], h["train_loss"], color=color, linewidth=2, linestyle=":")
    ax.plot(h["epoch"], h["val_loss"], color=color, linewidth=2)
    ax.axhline(h["val_entropy"].iloc[0], color=INK_MUTED, linewidth=1, linestyle="--")
    ax.scatter([b["epoch"]], [b["val_loss"]], s=50, color=color, edgecolors=SURFACE, linewidths=2, zorder=3)
    ax.set_title(label, loc="left", color=INK, fontsize=11)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    style(ax)
for ax in axes[1]:
    ax.set_xlabel("Epoch", color=INK_MUTED)
for ax in axes[:, 0]:
    ax.set_ylabel("Cross-entropy [nats]", color=INK_MUTED)
handles = [Line2D([0], [0], color=INK_MUTED, linewidth=2, linestyle=":", label="train CE"),
           Line2D([0], [0], color=INK_MUTED, linewidth=2, label="val CE"),
           Line2D([0], [0], color=INK_MUTED, linewidth=1, linestyle="--", label="val target entropy"),
           BEST_HANDLE]
finish(fig, "Train vs. validation cross-entropy", handles, "train_vs_val_ce")

## Top-k accuracy

Share of validation states where the Tutor's action is the argmax (top-1) or among the 5 / 20 highest
logits. For soft labels the Tutor's action is the argmax of the target, so all datasets are scored
against the same action.

In [ ]:
TOPK_COLUMNS = {"val_accuracy": "Top-1", "val_top5": "Top-5", "val_top20": "Top-20"}
fig, axes = plt.subplots(1, len(TOPK_COLUMNS), figsize=(14, 3.8), facecolor=SURFACE, sharex=True)
for ax, (column, title) in zip(axes, TOPK_COLUMNS.items()):
    for label, h in histories.items():
        if column not in h:  # runs from before top-20 was logged
            continue
        color, b = DATASET_COLORS[label], best[label]
        ax.plot(h["epoch"], h[column], color=color, linewidth=2)
        ax.scatter([b["epoch"]], [b[column]], s=50, color=color, edgecolors=SURFACE, linewidths=2, zorder=3)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel("Epoch", color=INK_MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    style(ax)
axes[0].set_ylabel("Validation accuracy", color=INK_MUTED)
finish(fig, "Top-k accuracy per epoch", dataset_handles(BEST_HANDLE), "topk_per_epoch")

## Regret per epoch

Left: mean regret of the argmax, its simulated max rho minus that of the best valid action, over states
where the argmax is valid. The best valid action is the Tutor's action except on near-ties, where the
Tutor prefers do-nothing (within its margin); section B measures regret against the Tutor's action
itself. Right: share of states whose argmax is invalid (illegal or ending the episode in simulation).
These are left out of the regret, so a model drifting towards invalid actions would look better on the
left alone.

In [ ]:
fig, (ax_regret, ax_invalid) = plt.subplots(1, 2, figsize=(12, 4.2), facecolor=SURFACE)
for label, h in histories.items():
    color, b = DATASET_COLORS[label], best[label]
    for ax, column in ((ax_regret, "val_regret"), (ax_invalid, "val_invalid_rate")):
        ax.plot(h["epoch"], h[column], color=color, linewidth=2)
        ax.scatter([b["epoch"]], [b[column]], s=50, color=color, edgecolors=SURFACE, linewidths=2, zorder=3)
ax_regret.set_ylabel("Mean regret vs. best valid action [Δ max rho]", color=INK_MUTED)
ax_invalid.set_ylabel("Invalid argmax rate", color=INK_MUTED)
for ax in (ax_regret, ax_invalid):
    ax.set_xlabel("Epoch", color=INK_MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    style(ax)
finish(fig, "Regret and invalid choices per epoch", dataset_handles(BEST_HANDLE), "regret_per_epoch")

# B. Selected model (validation, per state)

## Top-k curve

Left: share of states where the Tutor's action is among the k highest logits, for every k. Right: share
of states where at least one of the k highest logits is a *good* action (simulated max rho within the
good-action tolerance of the best valid action), over states with a valid action. The right panel is
what top-k simulation needs: a good action somewhere in the shortlist, not necessarily the Tutor's.
Vertical lines: the k of the top-k rollouts.

In [ ]:
ks = np.arange(1, N_ACTIONS + 1)


def share_at_most(ranks: np.ndarray, k: np.ndarray) -> np.ndarray:
    """Share of `ranks` that are <= each k."""
    return np.searchsorted(np.sort(ranks), k, side="right") / len(ranks)


topk_curves = {label: share_at_most(df["tutor_rank"].to_numpy(), ks) for label, df in states.items()}
good_curves = {label: share_at_most(df.loc[np.isfinite(df["rho_best"]), "good_rank"].to_numpy(), ks)
               for label, df in states.items()}

fig, (ax_tutor, ax_good) = plt.subplots(1, 2, figsize=(12, 4.2), facecolor=SURFACE, sharey=True)
for ax, curves in ((ax_tutor, topk_curves), (ax_good, good_curves)):
    for k in K_MARKS:
        ax.axvline(k, color=GRID, linewidth=1, zorder=0)
    for label, curve in curves.items():
        ax.plot(ks, curve, color=DATASET_COLORS[label], linewidth=2)
    ax.set_xscale("log")
    ax.set_xticks(K_MARKS + (N_ACTIONS,), [str(k) for k in K_MARKS + (N_ACTIONS,)])
    ax.set_xlabel("k (log scale)", color=INK_MUTED)
    ax.set_ylim(0, 1.02)
    style(ax)
ax_tutor.set_ylabel("Share of states", color=INK_MUTED)
ax_tutor.set_title("Tutor's action in top k", loc="left", color=INK, fontsize=11)
ax_good.set_title(f"Good action (≤ {TOLERANCE:g} above best) in top k", loc="left", color=INK, fontsize=11)
finish(fig, "Top-k curves of the selected models", dataset_handles(), "topk_curve")

idx = np.array(K_MARKS) - 1
pd.concat({"Tutor's action": pd.DataFrame({label: c[idx] for label, c in topk_curves.items()}, index=K_MARKS).T,
           "good action": pd.DataFrame({label: c[idx] for label, c in good_curves.items()}, index=K_MARKS).T},
          axis=1).round(3)

## Regret against the Tutor's action

Regret = simulated max rho of the argmax minus that of the Tutor's action, over states where the
Tutor's action is valid. Left: distribution of the regret where the argmax is a valid action other than
the Tutor's (the size of the misses). It can be negative where the model picks a topology action that
the Tutor turned down for do-nothing on a near-tie. Right: all states split into exact match, a
different action within the tolerance, a worse one, and an invalid one. Together they show whether the
mean regret comes from many small misses or few large ones.

In [ ]:
def miss_regret(df: pd.DataFrame) -> np.ndarray:
    return np.sort(df.loc[df["tutor_valid"] & df["pred_valid"] & ~df["match"], "regret"].to_numpy())


def outcome_shares(df: pd.DataFrame) -> pd.Series:
    df = df[df["tutor_valid"]]
    other = df["pred_valid"] & ~df["match"]
    return pd.Series({
        "Tutor's action": df["match"].mean(),
        f"other, ≤ {TOLERANCE:g} worse": (other & (df["regret"] <= TOLERANCE)).mean(),
        f"other, > {TOLERANCE:g} worse": (other & (df["regret"] > TOLERANCE)).mean(),
        "invalid": (~df["pred_valid"]).mean(),
    })


misses = {label: miss_regret(df) for label, df in states.items()}
outcomes = pd.DataFrame({label: outcome_shares(df) for label, df in states.items()}).T
OUTCOME_COLORS = ["#1baf7a", "#a3d9c4", "#eb6834", "#52514e"]

fig, (ax_ecdf, ax_bar) = plt.subplots(1, 2, figsize=(13, 4.2), facecolor=SURFACE, width_ratios=(1.2, 1))
for label, r in misses.items():
    ax_ecdf.step(r, np.arange(1, len(r) + 1) / len(r), where="post", color=DATASET_COLORS[label], linewidth=2)
ax_ecdf.axvline(0, color=INK_MUTED, linewidth=1)
ax_ecdf.axvline(TOLERANCE, color=INK_MUTED, linewidth=1, linestyle="--")
upper = np.quantile(np.concatenate(list(misses.values())), 0.95)
ax_ecdf.set_xlim(min(r.min() for r in misses.values()), upper)
ax_ecdf.set_xlabel("Regret vs. Tutor's action [Δ max rho] (x cut at 95th percentile)", color=INK_MUTED)
ax_ecdf.set_ylabel("Share of misses (valid, ≠ Tutor)", color=INK_MUTED)
style(ax_ecdf)

y = np.arange(len(outcomes))[::-1]
left = np.zeros(len(outcomes))
for column, color in zip(outcomes.columns, OUTCOME_COLORS):
    ax_bar.barh(y, outcomes[column], left=left, color=color, edgecolor=SURFACE, linewidth=1.5, label=column)
    left += outcomes[column].to_numpy()
ax_bar.set_yticks(y, outcomes.index, color=INK)
ax_bar.set_xlim(0, 1)
ax_bar.set_xlabel("Share of states", color=INK_MUTED)
ax_bar.legend(frameon=False, ncols=2, loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=9, labelcolor=INK)
style(ax_bar)
ax_bar.yaxis.grid(False)
ax_bar.xaxis.grid(True, color=GRID, linewidth=0.8)
handles = dataset_handles(Line2D([0], [0], color=INK_MUTED, linewidth=1, linestyle="--", label="tolerance"))
finish(fig, "Regret of the selected models against the Tutor", handles, "regret_distribution")
outcomes.round(3)

## Confidence

Left: distribution of the probability the policy gives the Tutor's action (log scale, the lowest 0.5 %
folded into the first bin; the dotted line is the uniform policy, 1/n_actions). Soft-label models are trained to spread mass over near-equivalent
actions, so they should give the Tutor's action less but its runners-up more. Right: reliability
diagram, the top-1 accuracy of states binned by the probability of the argmax. On the diagonal the
probabilities are calibrated; below it the model is overconfident. ECE: expected calibration error,
the state-weighted mean gap. Relevant when the policy's probabilities drive sampling or action masks
in RL.

In [ ]:
CONF_BINS = np.linspace(0, 1, 11)


def reliability(df: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    """Mean confidence and accuracy per bin of p(argmax), and the expected calibration error."""
    bins = pd.cut(df["p_pred"], CONF_BINS, include_lowest=True)
    table = df.groupby(bins, observed=True).agg(confidence=("p_pred", "mean"), accuracy=("match", "mean"),
                                                 n=("match", "size"))
    ece = float((table["n"] * (table["confidence"] - table["accuracy"]).abs()).sum() / len(df))
    return table, ece


calibration = {label: reliability(df) for label, df in states.items()}
# Bins from the 0.5th percentile up; the few states below land in the first bin.
p_low = min(np.quantile(df["p_tutor"], 0.005) for df in states.values())
p_bins = np.logspace(np.log10(p_low), 0, 31)

fig, (ax_hist, ax_rel) = plt.subplots(1, 2, figsize=(12, 4.4), facecolor=SURFACE)
for label, df in states.items():
    color = DATASET_COLORS[label]
    ax_hist.hist(df["p_tutor"].clip(lower=p_low), bins=p_bins, weights=np.full(len(df), 1 / len(df)), histtype="step",
                 color=color, linewidth=2)
    table, ece = calibration[label]
    table = table[table["n"] >= 20]  # too few states for a stable accuracy
    ax_rel.plot(table["confidence"], table["accuracy"], color=color, linewidth=2, marker="o", markersize=4,
                label=f"{label}  ECE {ece:.3f}")
ax_hist.axvline(1 / N_ACTIONS, color=INK_MUTED, linewidth=1, linestyle=":")
ax_hist.set_xscale("log")
ax_hist.set_xlabel("p(Tutor's action) (log scale)", color=INK_MUTED)
ax_hist.set_ylabel("Share of states", color=INK_MUTED)
ax_rel.plot([0, 1], [0, 1], color=INK_MUTED, linewidth=1, linestyle="--")
ax_rel.set_xlim(0, 1)
ax_rel.set_ylim(0, 1)
ax_rel.set_xlabel("p(argmax), bin mean", color=INK_MUTED)
ax_rel.set_ylabel("Top-1 accuracy in bin", color=INK_MUTED)
ax_rel.legend(frameon=False, fontsize=9, labelcolor=INK, loc="upper left")
for ax in (ax_hist, ax_rel):
    style(ax)
handles = dataset_handles(Line2D([0], [0], color=INK_MUTED, linewidth=1, linestyle=":", label="uniform"),
                          Line2D([0], [0], color=INK_MUTED, linewidth=1, linestyle="--", label="calibrated"))
finish(fig, "Confidence of the selected models", handles, "confidence")

## Summary

Selected model per dataset on the validation split. Regret is against the Tutor's action, over states
where both actions are valid; good action rate: argmax within the tolerance of the best valid action.

In [ ]:
def summary_row(label: str) -> dict[str, float]:
    df = states[label]
    both_valid = df["tutor_valid"] & df["pred_valid"]
    return {
        "selected epoch": int(best[label]["epoch"]),
        "KL": best[label]["val_kl"],
        **{f"top-{k}": (df["tutor_rank"] <= k).mean() for k in (1, 5, 20)},
        "good action rate": (df.loc[np.isfinite(df["rho_best"]), "good_rank"] == 1).mean(),
        "regret vs. Tutor": df.loc[both_valid, "regret"].mean(),
        "invalid rate": (~df["pred_valid"]).mean(),
        "mean p(Tutor)": df["p_tutor"].mean(),
        "ECE": calibration[label][1],
    }


pd.DataFrame({label: summary_row(label) for label in RUNS}).T.round(4)

# C. Hyperparameter search

One Optuna study per dataset (`experiments/pretrain_hpo.py`, search space in
`configs/rllib/pretrain_hpo.yaml`) minimises the validation KL. Trials per state: complete trials were
trained until early stopping, pruned ones were stopped by the pruner after a poor epoch.

In [ ]:
studies = {label: open_study(path, path.name) for label, path in STUDIES.items()}
trials = pd.concat([pd.read_csv(path / TRIALS_FILE).assign(dataset=label) for label, path in STUDIES.items()],
                   ignore_index=True)
trials.groupby(["dataset", "state"], sort=False).size().unstack(fill_value=0)

## Optimization history

Every trial's validation KL (filled: complete, hollow: pruned at its last reported epoch) and the
best complete value so far. KL is comparable between datasets in units (nats), but each study fits
its own targets.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), facecolor=SURFACE)
for ax, (label, group) in zip(axes.flat, trials.groupby("dataset", sort=False)):
    color = DATASET_COLORS[label]
    group = group.sort_values("number")
    done, pruned = group[group["state"] == "COMPLETE"], group[group["state"] == "PRUNED"]
    ax.scatter(pruned["number"], pruned["value"], s=40, facecolors="none", edgecolors=color, linewidths=1.2)
    ax.scatter(done["number"], done["value"], s=40, color=color, edgecolors=SURFACE, linewidths=1.5)
    best_so_far = done.set_index("number")["value"].cummin().reindex(group["number"]).ffill()
    ax.plot(best_so_far.index, best_so_far.values, color=INK_MUTED, linewidth=2, drawstyle="steps-post")
    ax.set_title(f"{label}  (best {done['value'].min():.3f})", loc="left", color=INK, fontsize=11)
    ax.set_xlabel("Trial", color=INK_MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylabel("Validation KL [nats]", color=INK_MUTED)
    style(ax)
handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=INK_MUTED, markeredgecolor=INK_MUTED, label="complete"),
           Line2D([0], [0], marker="o", color="none", markerfacecolor="none", markeredgecolor=INK_MUTED, label="pruned"),
           Line2D([0], [0], color=INK_MUTED, linewidth=2, label="best so far")]
finish(fig, "Junior HPO: validation KL per trial", handles, "optimization_history")

## Hyperparameter importance

fANOVA importance (`optuna.importance.get_param_importances`) of each hyperparameter for the
validation KL, per study and over complete trials. Importances sum to 1 within a study.

In [ ]:
importance = pd.DataFrame({label: optuna.importance.get_param_importances(study)
                           for label, study in studies.items()}).fillna(0.0)
importance.index = [name.split(".", 1)[-1] for name in importance.index]
importance = importance.loc[importance.mean(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 6), facecolor=SURFACE)
height = 0.8 / len(STUDIES)
y = np.arange(len(importance))
for i, label in enumerate(STUDIES):
    ax.barh(y - (i - (len(STUDIES) - 1) / 2) * height, importance[label], height,
            color=DATASET_COLORS[label], edgecolor=SURFACE, linewidth=1, label=label)
ax.set_yticks(y, importance.index)
ax.set_xlabel("Importance (fANOVA)", color=INK_MUTED)
ax.set_title("Hyperparameter importance for validation KL", loc="left", color=INK, fontsize=12)
ax.legend(frameon=False, labelcolor=INK, loc="lower right")
style(ax)
ax.yaxis.grid(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
fig.tight_layout()
save(fig, "param_importance")
plt.show()

## Best trial per dataset

Hyperparameters of the best complete trial of each study (lowest validation KL), one column per
dataset. `search space`: the values the study sampled from (`{…}` categorical, `[low, high]`
uniform or log-uniform). `mean importance`: fANOVA importance averaged over the four studies;
hyperparameters are sorted by it.

In [ ]:
def format_space(spec: dict) -> str:
    """Search space of one hyperparameter as written in the HPO config."""
    if spec["type"] == "categorical":
        return "{" + ", ".join(str(choice) for choice in spec["choices"]) + "}"
    return f"[{spec['low']:g}, {spec['high']:g}]" + (" (log)" if spec.get("log") else "")


def format_value(value) -> str:
    return f"{value:.3g}" if isinstance(value, float) else str(value)


search_space = yaml.safe_load(HPO_CONFIG.read_text())["hpo"]["search_space"]
best_rows = (trials[trials["state"] == "COMPLETE"].sort_values("value")
             .groupby("dataset", sort=False).head(1).set_index("dataset").loc[list(STUDIES)])

trial_info = best_rows[["number", "user_attrs_best_epoch", "user_attrs_n_epochs"]].astype(int).set_axis(
    ["trial", "best epoch", "epochs run"], axis=1).T
hyperparams = best_rows[[f"params_{name}" for name in search_space]].set_axis(
    [name.split(".", 1)[-1] for name in search_space], axis=1).T.map(format_value)
hyperparams["search space"] = [format_space(spec) for spec in search_space.values()]
hyperparams["mean importance"] = importance.mean(axis=1).round(3)
hyperparams = hyperparams.sort_values("mean importance", ascending=False)

pd.concat([trial_info, hyperparams]).fillna("").rename_axis(columns=None)